# RapidFire AI RAG Experiment - Documentation Q&A Chatbot

Evaluates **8** RAG configs in one `run_evals` pass (chunk 128/512, overlap 16, similarity `k` 10/20, rerank `top_n` 2/5).

## Load API Key

In [1]:
from pathlib import Path
import os

# Get API Key
TRITON_API_KEY = Path("~/api-key.txt").expanduser().read_text(encoding="utf-8").splitlines()[0].strip()

# Set environment variables
os.environ.setdefault("OPENAI_API_KEY", TRITON_API_KEY)
os.environ.setdefault("JUDGE_BASE_URL", "https://tritonai-api.ucsd.edu/v1")
os.environ.setdefault("JUDGE_MODEL", "claude-sonnet-4-6-aws")

'claude-sonnet-4-6-aws'

## Import Required Libraries

In [2]:
# RapidFireAI Imports
from rapidfireai.automl import (
    List,
    RFLangChainRagSpec,
    RFOpenAIAPIModelConfig,
    RFPromptManager,
    RFGridSearch,
)
from rapidfireai import Experiment

# Standard library imports
import re
import json
import itertools
from typing import List as listtype, Dict, Any, Optional
from pathlib import Path
import tiktoken

# Data and ML imports
import pandas as pd
from datasets import Dataset

# LangChain imports
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_openai import OpenAIEmbeddings

# Import evaluation functions
from project1_eval import call_judge, f1_at_k, precision_at_k, recall_at_k, to_spans
from rapidfire_integration_example import sample_compute_metrics_fn, sample_accumulate_metrics_fn

INFO 05-03 21:01:10 [importing.py:44] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.
INFO 05-03 21:01:10 [importing.py:68] Triton not installed or not compatible; certain GPU-related functions will not be available.


## Load Dataset

Load the validation set golden Q&A pairs and experiment.json file.

In [3]:
# Load input JSON file
input_file = "validation-set-golden-qa-pairs.json"
output_file = "8_output.json"

with open(input_file, "r") as f:
    data = json.load(f)

# Build dataset rows
rows = [
    {
        "query_id": int(entry["question_id"]),          
        "query": str(entry["question"]),
        "reference_answer": str(entry.get("reference_answer", "")), 
        "source_evidence": entry.get("source_evidence", []),
    }
    for entry in data
]

dataset = Dataset.from_list(rows)
print(f"Loaded {len(dataset)} examples from {input_file}")
print(f"Dataset preview:")
print(dataset[0])

Loaded 45 examples from validation-set-golden-qa-pairs.json
Dataset preview:
{'query_id': 1, 'query': 'What are the two knob set generators currently supported by RapidFire AI for creating multi-config specifications?', 'reference_answer': "RapidFire AI currently supports two knob set generators: List() for a discrete set of values and Range() for sampling from a continuous value interval. List() takes a list of discrete values where all values must be the same Python data type. Range() takes a start, end, and dtype parameter (either 'int' or 'float') and performs uniform sampling within the given interval.", 'source_evidence': [{'file': 'configs.rst', 'lines': [23, 48]}]}


## Create Experiment

In [4]:
experiment = Experiment(experiment_name="8_overlap", mode="evals")

Experiment 8_overlap created with Experiment ID: 11 at /home/ayliang/rapidfireai/rapidfire_experiments/8_overlap
Created directory: /home/ayliang/rapidfireai/logs/8_overlap


## Define RAG Configuration

Full factorial **8 runs**: chunk size (128, 512), overlap **16**, similarity `k` (10, 20), reranker `top_n` (2, 5). Chunking uses tiktoken **gpt2** (BPE); embeddings are **api-tgpt-embeddings**. MMR is out of scope for this grid.

In [5]:
# Grid axes (2×2×2 = 8 configs): chunk size, similarity k, reranker top_n
CHUNK_SIZES = [128, 512]
CHUNK_OVERLAP = 8
SEARCH_KS = [10, 20]
RERANK_TOP_N = [2, 5]
batch_size = 32


def build_rag_for_chunking(
    chunk_size: int,
    chunk_overlap: int,
    retrieve_k: int,
    rerank_top_n: int,
) -> RFLangChainRagSpec:
    """One fully-specified RAG config (scalar hyperparameters per grid point)."""
    return RFLangChainRagSpec(
        document_loader=DirectoryLoader(
            path="sourcedocs/sourcedocs/",
            glob="**/*.rst",
            loader_cls=TextLoader,
            loader_kwargs={"encoding": "utf-8"},
            sample_seed=1337,
        ),
        text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            encoding_name="gpt2",
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            add_start_index=True,
        ),
        embedding_cfg={
            "class": OpenAIEmbeddings,
            "model": "api-tgpt-embeddings",
            "api_key": TRITON_API_KEY,
            "base_url": "https://tritonai-api.ucsd.edu",
            "check_embedding_ctx_length": False,
        },
        vector_store_cfg={"type": "faiss", "batch_size": batch_size},
        search_cfg={"type": "similarity", "k": retrieve_k},
        reranker_cfg=List([
            {
                "class": CrossEncoderReranker,
                "model_name": "BAAI/bge-reranker-v2-m3",
                "model_kwargs": {"device": "cpu"},
                "top_n": rerank_top_n,
            }
        ]),
        enable_gpu_search=False,
    )


_n = len(CHUNK_SIZES) * len(SEARCH_KS) * len(RERANK_TOP_N)
print(
    f"RAG builder ready: chunks={CHUNK_SIZES}, overlap={CHUNK_OVERLAP}, "
    f"search_k={SEARCH_KS}, rerank_top_n={RERANK_TOP_N} → {_n} grid points"
)

RAG builder ready: chunks=[128, 512], overlap=8, search_k=[10, 20], rerank_top_n=[2, 5] → 8 grid points


## Define Instructions and Helper Functions

In [6]:
INSTRUCTIONS = """You are a precise technical assistant for the RapidFire AI documentation.
You will be given a user question and relevant context chunks retrieved from the RapidFire AI docs.

Rules:
- Answer using ONLY information present in the provided context. Do not use outside knowledge.
- Be specific and complete — include parameter names, types, defaults, and exact values when present.
- For procedural questions, list the steps in order.
- For comparative questions, clearly distinguish between the two things being compared.
- For factual/lookup questions, give the exact answer directly.
- If the context does not contain enough information to answer, say: "The provided context does not contain enough information to answer this question."
- Do not add caveats, filler phrases, or unnecessary preamble. Get to the answer immediately.
- Stay within 2000 tokens total context budget.

Respond with your answer only. No reasoning prefix needed.
Question: "What are the two main execution functions provided by the Experiment class for launching workflows?"
Answer: "The two main execution functions are run_fit() for training/evaluation workflows and run_evals() for LLM evaluation workflows."
Source Evidence: source_evidence": [
    { "file": "experiment.rst", "lines": [69, 75] },
    { "file": "experiment.rst", "lines": [154, 160] }
    ]
"""

# Token-aware truncation
MAX_TOKENS_PER_QUERY: int = 2000
SAFETY_TOKENS: int = 50

def _truncate_context(text: str, max_chars: Optional[int]) -> str:
    if max_chars is None or len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n\n[context truncated]"

def _truncate_context_by_tokens(text: str, max_tokens: Optional[int], encoding) -> str:
    if max_tokens is None:
        return text
    if max_tokens <= 0:
        return ""
    if encoding is None:
        return _truncate_context(text, max_chars=int(max_tokens * 4))
    toks = encoding.encode(text)
    if len(toks) <= max_tokens:
        return text
    try:
        return encoding.decode(toks[:max_tokens]) + "\n\n[context truncated]"
    except Exception:
        try:
            return "".join(toks[:max_tokens]) + "\n\n[context truncated]"
        except Exception:
            return "\n\n[context truncated]"

def chunk_to_lines(doc: Document) -> listtype:
    """Convert a chunk's character `start_index` into [start_line, end_line]."""
    src = doc.metadata["source"]
    start_idx = doc.metadata["start_index"]
    text = Path(src).read_text(encoding="utf-8")
    start_line = text[:start_idx].count("\n") + 1
    end_line = start_line + (doc.page_content or "").count("\n")
    if end_line < start_line:
        end_line = start_line
    return [start_line, end_line]

## Define Preprocessing and Postprocessing Functions

In [7]:
# Initialize output storage
output_rows = []
output_rows_jsonl = Path(output_file + ".rows.jsonl")
if output_rows_jsonl.exists():
    output_rows_jsonl.unlink()

def openai_sample_preprocess_fn(
    batch: Dict[str, listtype], rag: RFLangChainRagSpec, prompt_manager: RFPromptManager
) -> Dict[str, listtype]:
    """Function to prepare the final inputs given to the generator model"""

    all_context = rag.get_context(batch_queries=batch["query"], serialize=False)
    serialized_context = rag.serialize_documents(all_context)
    
    # Token-aware per-query truncation
    encoding = tiktoken.get_encoding("gpt2")
    system_tokens = len(encoding.encode(INSTRUCTIONS or ""))
    template_tokens = len(encoding.encode("\nQuestion:\n\nContext:\n\nAnswer:"))
    new_serialized = []
    
    for question, ctx in zip(batch.get("query", []), serialized_context):
        q_tokens = len(encoding.encode(question or ""))
        avail = MAX_TOKENS_PER_QUERY - (system_tokens + q_tokens + template_tokens + SAFETY_TOKENS)
        if avail <= 0:
            new_serialized.append("")
        else:
            new_serialized.append(_truncate_context_by_tokens(ctx, avail, encoding))
    
    serialized_context = new_serialized
    batch["query_id"] = [int(query_id) for query_id in batch["query_id"]]

    batch["ground_truth_spans"] = [
        [
            (item["file"], int(item["lines"][0]), int(item["lines"][1]))
            for item in evidence
            if item.get("file") and item.get("lines") and len(item["lines"]) >= 2
        ]
        for evidence in batch.get("source_evidence", [])
    ]

    per_doc_lines = [[chunk_to_lines(doc) for doc in docs] for docs in all_context]

    return {
        "prompts": [
            [
                {"role": "system", "content": INSTRUCTIONS},
                {
                    "role": "user",
                    "content": f"\nQuestion:\n{question}\n\nContext:\n{context}\n\nAnswer:"
                },
            ]
            for question, context in zip(batch["query"], serialized_context)
        ],
        "serialized_context": serialized_context,
        "retrieved_context": serialized_context,
        "sources": [
            [
                {"file": Path(doc.metadata["source"]).name, "lines": lines}
                for doc, lines in zip(docs, doc_lines)
            ]
            for docs, doc_lines in zip(all_context, per_doc_lines)
        ],
        "retrieved_spans": [
            [
                (Path(doc.metadata["source"]).name, lines[0], lines[1])
                for doc, lines in zip(docs, doc_lines)
            ]
            for docs, doc_lines in zip(all_context, per_doc_lines)
        ],
        **batch,
    }

def sample_postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    """Postprocess outputs produced by generator model"""
    batch["answer"] = batch["generated_text"]

    for qid, ans, ctx, srcs in zip(
        batch["query_id"],
        batch["answer"],
        batch["retrieved_context"],
        batch["sources"],
    ):
        row = {
            "question_id": int(qid),
            "answer": ans,
            "retrieved_context": ctx,
            "sources": srcs,
        }
        output_rows.append(row)
        # Persist each row so outputs survive multi-process Ray workers
        with open(output_rows_jsonl, "a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    return batch

## Define Generator Configuration

In [8]:
# Exactly 8 configs: chunk × similarity-k × rerank top_n (no reliance on nested List() expansion)
openai_configs = []
for chunk_size, retrieve_k, rerank_top_n in itertools.product(
    CHUNK_SIZES, SEARCH_KS, RERANK_TOP_N
):
    openai_configs.append(
        RFOpenAIAPIModelConfig(
            client_config={
                "api_key": TRITON_API_KEY,
                "base_url": "https://tritonai-api.ucsd.edu",
                "max_retries": 2,
            },
            model_config={
                "model": "api-mistral-small-3.2-2506",
                "max_completion_tokens": 2048,
            },
            rpm_limit=120,
            tpm_limit=1_000_000,
            rag=build_rag_for_chunking(
                chunk_size=chunk_size,
                chunk_overlap=CHUNK_OVERLAP,
                retrieve_k=retrieve_k,
                rerank_top_n=rerank_top_n,
            ),
            prompt_manager=None,
        )
    )

assert len(openai_configs) == 8

config_set = {
    "openai_config": List(openai_configs),
    "batch_size": batch_size,
    "preprocess_fn": openai_sample_preprocess_fn,
    "postprocess_fn": sample_postprocess_fn,
    "compute_metrics_fn": sample_compute_metrics_fn,
    "accumulate_metrics_fn": sample_accumulate_metrics_fn,
}

config_group = RFGridSearch(config_set)

## Run Evaluation

In [9]:
# Launch evals
results = experiment.run_evals(
    config_group=config_group,
    dataset=dataset,
    num_shards=4,
    num_actors=4,
    seed=42,
)

print("Evaluation completed!")

=== Preprocessing RAG Sources ===


RAG Source ID,Status,Duration,Device,Vector Store
1,Complete,32.3s,CPU,FAISS
2,Complete,32.3s,CPU,FAISS



=== Multi-Config Experiment Progress ===


Run ID,Model,Status,Progress,Conf. Interval,text_splitter,chunk_size,chunk_overlap,embedding_cfg.base_url,embedding_cfg.check_embedding_ctx_length,embedding_cfg.class,embedding_cfg.model,vector_store_cfg.batch_size,vector_store_cfg.type,search_cfg.k,search_cfg.type,reranker_cfg.class,reranker_cfg.model_name,reranker_cfg.top_n,model_config,Completeness_normalized,Correctness_pass_rate,F1_at_5,Faithfulness_pass_rate,Generation_Score_3_released,Judge Failures,Precision_at_5,Processing Time,Recall_at_5,Retrieval Score,Samples Per Second,Samples Processed,Throughput,Total
1,api-mistral-small-3.2-2506,COMPLETED,4/4,0.000,RecursiveCharacterTextSplitter,128,8,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,api-tgpt-embeddings,32,faiss,10,similarity,CrossEncoderReranker,BAAI/bge-reranker-v2-m3,2,max_completion_tokens=2048,"0.3822 [0.3822, 0.3822]","0.5111 [0.5111, 0.5111]","0.6089 [0.6089, 0.6089]","0.8444 [0.8444, 0.8444]","0.5793 [0.5793, 0.5793]",2.00,"0.5333 [0.5333, 0.5333]",4197.70 seconds,"0.7944 [0.7944, 0.7944]","0.6456 [0.6456, 0.6456]",0.01,45,0.0/s,45
2,api-mistral-small-3.2-2506,COMPLETED,4/4,0.000,RecursiveCharacterTextSplitter,128,8,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,api-tgpt-embeddings,32,faiss,10,similarity,CrossEncoderReranker,BAAI/bge-reranker-v2-m3,5,max_completion_tokens=2048,"0.4844 [0.4844, 0.4844]","0.6667 [0.6667, 0.6667]","0.4786 [0.4786, 0.4786]","0.8889 [0.8889, 0.8889]","0.6800 [0.6800, 0.6800]",2.00,"0.3467 [0.3467, 0.3467]",4174.69 seconds,"0.8944 [0.8944, 0.8944]","0.5732 [0.5732, 0.5732]",0.01,45,0.0/s,45
3,api-mistral-small-3.2-2506,COMPLETED,4/4,0.000,RecursiveCharacterTextSplitter,128,8,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,api-tgpt-embeddings,32,faiss,20,similarity,CrossEncoderReranker,BAAI/bge-reranker-v2-m3,2,max_completion_tokens=2048,"0.3333 [0.3333, 0.3333]","0.4889 [0.4889, 0.4889]","0.6593 [0.6593, 0.6593]","0.7778 [0.7778, 0.7778]","0.5333 [0.5333, 0.5333]",6.00,"0.5778 [0.5778, 0.5778]",4172.49 seconds,"0.8444 [0.8444, 0.8444]","0.6938 [0.6938, 0.6938]",0.01,45,0.0/s,45
4,api-mistral-small-3.2-2506,COMPLETED,4/4,0.000,RecursiveCharacterTextSplitter,128,8,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,api-tgpt-embeddings,32,faiss,20,similarity,CrossEncoderReranker,BAAI/bge-reranker-v2-m3,5,max_completion_tokens=2048,"0.4622 [0.4622, 0.4622]","0.6000 [0.6000, 0.6000]","0.4918 [0.4918, 0.4918]","0.8667 [0.8667, 0.8667]","0.6430 [0.6430, 0.6430]",4.00,"0.3556 [0.3556, 0.3556]",4170.41 seconds,"0.9167 [0.9167, 0.9167]","0.5880 [0.5880, 0.5880]",0.01,45,0.0/s,45
5,api-mistral-small-3.2-2506,COMPLETED,4/4,0.000,RecursiveCharacterTextSplitter,512,8,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,api-tgpt-embeddings,32,faiss,10,similarity,CrossEncoderReranker,BAAI/bge-reranker-v2-m3,2,max_completion_tokens=2048,"0.6178 [0.6178, 0.6178]","0.8222 [0.8222, 0.8222]","0.6000 [0.6000, 0.6000]","0.9333 [0.9333, 0.9333]","0.7911 [0.7911, 0.7911]",0.0000,"0.5000 [0.5000, 0.5000]",3938.57 seconds,"0.8056 [0.8056, 0.8056]","0.6352 [0.6352, 0.6352]",0.01,45,0.0/s,45
6,api-mistral-small-3.2-2506,COMPLETED,4/4,0.000,RecursiveCharacterTextSplitter,512,8,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,api-tgpt-embeddings,32,faiss,10,similarity,CrossEncoderReranker,BAAI/bge-reranker-v2-m3,5,max_completion_tokens=2048,"0.6044 [0.6044, 0.6044]","0.8000 [0.8000, 0.8000]","0.3962 [0.3962, 0.3962]","0.9111 [0.9111, 0.9111]","0.7719 [0.7719, 0.7719]",2.00,"0.2622 [0.2622, 0.2622]",3922.52 seconds,"0.9056 [0.9056, 0.9056]","0.5213 [0.5213, 0.5213]",0.01,45,0.0/s,45
7,api-mistral-small-3.2-2506,COMPLETED,4/4,0.000,RecursiveCharacterTextSplitter,512,8,https://tritonai-api.ucsd.edu,False,OpenAIEmbeddings,api-tgpt-embeddings,32,faiss,20,similarity,CrossEncoderReranker,BAAI/bge-reranker-v2-m3,2,max_completion_tokens=2048,"0.5911 [0.5911, 0.5911]","0.7333 [0.7333, 0.7333]","0.6296 [0.6296, 0.6296]","0.8444 [0.8444, 0.8444]","0.7230 [0.7230, 0.7230]",4.00,"0.5333 [0.5333, 0.53

Evaluation completed!


## Save Output

In [10]:
# Write final output JSON
output_rows.sort(key=lambda x: x["question_id"])

if output_rows_jsonl.exists():
    with open(output_rows_jsonl, "r", encoding="utf-8") as f:
        output_rows = [json.loads(line) for line in f if line.strip()]
    output_rows.sort(key=lambda x: x["question_id"])

with open(output_file, "w") as f:
    json.dump(output_rows, f, indent=2)

print(f"Output saved to {output_file}")
print(f"Total examples processed: {len(output_rows)}")

Output saved to 8_output.json
Total examples processed: 360


## End Experiment

In [11]:
experiment.end()
print("Experiment ended.")

Experiment 8_overlap ended
Experiment ended.


## View Experiment Logs

In [12]:
# Get the experiment-specific log file
log_file = experiment.get_log_file_path()

print(f"📄 Log File: {log_file}")
print()

if log_file.exists():
    print("=" * 80)
    print(f"Last 30 lines of {log_file.name}:")
    print("=" * 80)
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            print(line.rstrip())
else:
    print(f"❌ Log file not found: {log_file}")

📄 Log File: /home/ayliang/rapidfireai/logs/8_overlap/rapidfire.log

Last 30 lines of rapidfire.log:
2026-05-03 22:12:08 | Controller | INFO | controller.py:695 | [8_overlap:Controller] Computing final metrics for all pipelines...
2026-05-03 22:12:08 | Experiment | INFO | metric_rfmetric_manager.py:190 | [8_overlap:Experiment] Ending run: 40cff43a61e0446e9bc546f69dce09bf, 1 in rf_mlflow
2026-05-03 22:12:08 | Experiment | WARNING | metric_mlflow_manager.py:101 | [8_overlap:Experiment] Run 40cff43a61e0446e9bc546f69dce09bf is not the active run, no local context to clear
2026-05-03 22:12:08 | Controller | INFO | controller.py:853 | [8_overlap:Controller] Pipeline 1 (Pipeline 1) completed successfully
2026-05-03 22:12:09 | Experiment | INFO | metric_rfmetric_manager.py:190 | [8_overlap:Experiment] Ending run: e1b65ff73b364d59b7eee9098455870f, 2 in rf_mlflow
2026-05-03 22:12:09 | Experiment | WARNING | metric_mlflow_manager.py:101 | [8_overlap:Experiment] Run e1b65ff73b364d59b7eee9098455870f